# V7_0_N03 — Quality Is a Decision Variable

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft. Illustrative data do not constitute official statistics or operational authorization.

## Learning outcomes
Measure fitness for use across relevance, accuracy, timeliness, accessibility, coherence, coverage, representativeness, revisions, and drift; route exceptions to accountable review.

In [1]:
import numpy as np,pandas as pd
rng=np.random.default_rng(703)
sectors=['Agriculture','Health','Education','Habitat']
df=pd.DataFrame({'sector':np.repeat(sectors,60),'value':rng.normal(100,15,240),'delay_days':rng.integers(0,45,240),'weight':rng.uniform(.5,2,240),'period':np.tile(np.repeat([1,2,3,4],15),4)})
df.loc[rng.choice(df.index,18,replace=False),'value']=np.nan
df.head()

## 1. Completeness is decision-specific
A missing value rate is not enough. Identify which fields are mandatory for which decision and whether missingness is concentrated in a subgroup, geography, or period.

In [2]:
overall=df.value.isna().mean(); by_sector=df.groupby('sector').value.apply(lambda x:x.isna().mean())
print('OVERALL',round(overall,3)); print(by_sector.round(3).to_string())

OVERALL 0.075
sector
Agriculture    0.050
Education      0.100
Habitat        0.067
Health         0.083


## 2. Timeliness versus decision horizon
Data can be accurate but arrive too late. The timeliness gate compares reporting delay with the period in which intervention remains possible.

In [3]:
horizon={'Agriculture':30,'Health':7,'Education':14,'Habitat':30}
df['horizon_days']=df.sector.map(horizon); df['timely']=df.delay_days<=df.horizon_days
print(df.groupby('sector').timely.mean().round(3).to_string())

sector
Agriculture    0.600
Education      0.283
Habitat        0.717
Health         0.167


## 3. Weighted coverage
Record counts can hide undercoverage when missing units represent larger populations. Compare unweighted and design/size-weighted coverage.

In [4]:
df['observed']=df.value.notna(); coverage=df.groupby('sector').apply(lambda g:pd.Series({'unweighted':g.observed.mean(),'weighted':np.average(g.observed,weights=g.weight)}),include_groups=False)
print(coverage.round(3).to_string())

             unweighted  weighted
sector                           
Agriculture       0.950     0.938
Education         0.900     0.913
Habitat           0.933     0.927
Health            0.917     0.913


## 4. Revisions and reproducibility
Official values may be revised. Preserve vintages, reasons, and dissemination status rather than silently overwriting earlier values.

In [5]:
vintages=pd.DataFrame({'indicator':['X']*3,'period':['2026Q1']*3,'vintage':['initial','revised','final'],'value':[98.2,101.4,100.9],'reason':['first release','late returns','quality closure']})
vintages['revision_from_initial']=vintages.value-vintages.value.iloc[0]
print(vintages.to_string(index=False))

indicator period vintage  value          reason  revision_from_initial
        X 2026Q1 initial   98.2   first release                    0.0
        X 2026Q1 revised  101.4    late returns                    3.2
        X 2026Q1   final  100.9 quality closure                    2.7


## 5. Drift is a review signal
A distribution change may represent real change, measurement change, coding change, or error. It triggers investigation; it does not diagnose the cause.

In [6]:
a=rng.normal(100,12,400); b=rng.normal(112,16,400)
bins=np.quantile(a,np.linspace(0,1,11)); bins[0],bins[-1]=-np.inf,np.inf
pa=np.histogram(a,bins)[0]/len(a); pb=np.histogram(b,bins)[0]/len(b); eps=1e-6
psi=np.sum((pb-pa)*np.log((pb+eps)/(pa+eps)))
print('PSI_REVIEW_SIGNAL',round(float(psi),3))

PSI_REVIEW_SIGNAL 0.767


## 6. Quality gate and exception workflow
A composite score helps routing but must not conceal failing critical dimensions. Critical failures override the average.

In [7]:
quality=pd.DataFrame({'sector':sectors,'completeness':[.94,.96,.91,.88],'timeliness':[.82,.71,.86,.79],'coherence':[.90,.87,.93,.84],'critical_key_pass':[True,True,True,False]})
quality['score']=quality[['completeness','timeliness','coherence']].mean(axis=1)
quality['status']=np.where(~quality.critical_key_pass,'STOP',np.where(quality.score>=.85,'PASS','REVIEW'))
print(quality.to_string(index=False))

     sector  completeness  timeliness  coherence  critical_key_pass    score status
Agriculture          0.94        0.82       0.90               True 0.886667   PASS
     Health          0.96        0.71       0.87               True 0.846667 REVIEW
  Education          0.91        0.86       0.93               True 0.900000   PASS
    Habitat          0.88        0.79       0.84              False 0.836667   STOP


## Exercises
1. Add a subgroup completeness check. 2. Explain why averaging can be unsafe. 3. Propose a revision policy. 4. Distinguish concept drift from data drift.

## Exact solutions
1. Group by the policy-relevant subgroup and compare both unweighted and weighted missingness with disclosure controls. 2. A high score can mask a failed key, unlawful source, or unusable delay; critical gates must override averages. 3. Preserve vintages, reasons, approval, release status, notification, and reproducible recalculation. 4. Data drift changes input distributions; concept drift changes the relationship between inputs and the target/outcome.

In [8]:
assert quality.loc[quality.sector=='Habitat','status'].item()=='STOP'
assert 0<=overall<=1 and psi>=0
print('V7_0_N03_COMPLETE_EXECUTION_PASS')

V7_0_N03_COMPLETE_EXECUTION_PASS
